Ячейка 2 — Setup

In [1]:
# %%
"""
SETUP
"""
import json
import re
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
import pandas as pd

STRUCTURED = Path("../data/structured_data")
DICT_PATH = Path("../data/dictionary/result/dictionary_en_ru.json")
OUT_DIR = Path("../data/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

GUN_TYPES = {"PS", "SR", "AR", "SG", "SM"}
RARITIES = {"legendary", "pearlescent"}

# --- микрословарики (сразу в DF по-русски) ---
RARITY_RU = {
    "legendary": "Легендарный",
    "pearlescent": "Перламутровый",
}

TYPE_RU = {
    "PS": "Пистолет",
    "SR": "Снайперская винтовка",
    "AR": "Штурмовая винтовка",
    "SG": "Дробовик",
    "SM": "Пистолет-пулемёт",
}

MANU_RU = {
    "JAK": "Джейкобс",
    "MAL": "Маливань",
    "TED": "Тедиор",
    "TOR": "Торг",
    "VLA": "Владоф",
    "ORD": "Орден",
    "DAD": "Дедалус",
    "BOR": "Риппер",
}

ELEMENT_RU = {
    "Fire": "Огонь",
    "Cryo": "Крио",
    "Shock": "Шок",
    "Radiation": "Радиация",
    "Corrosive": "Коррозия",
    "Kinetic": "Кинетический",
}

print("Setup OK")
print("STRUCTURED:", STRUCTURED.resolve())
print("DICT:", DICT_PATH.resolve())

Setup OK
STRUCTURED: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/structured_data
DICT: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/dictionary/result/dictionary_en_ru.json


Ячейка 3 — Словарь переводов

In [2]:
# %%
"""
DICTIONARY
guid → {en, ru}
en_lower → [ru, ...]
"""
with open(DICT_PATH, encoding="utf-8") as f:
    raw_dict = json.load(f)

guid_map = {}          # upper/lower guid → entry
en_to_ru = defaultdict(list)  # en.lower() → list of ru

for k, v in raw_dict.items():
    if not isinstance(v, dict):
        continue
    en = (v.get("en") or "").strip()
    ru = (v.get("ru") or "").strip()
    entry = {"en": en, "ru": ru, "source": v.get("source"), "namespace": v.get("namespace")}

    guid_map[k] = entry
    guid_map[k.upper()] = entry
    guid_map[k.lower()] = entry

    if en:
        en_to_ru[en.lower()].append(ru)

def translate_by_guid(guid: str | None) -> str:
    if not guid:
        return "-"
    e = guid_map.get(guid) or guid_map.get(guid.upper()) or guid_map.get(guid.lower())
    if not e:
        return "(перевод не найден)"
    ru = (e.get("ru") or "").strip()
    if not ru:
        return "(перевод не найден)"
    return ru

def translate_by_en(text: str | None) -> str:
    if not text or not str(text).strip():
        return "-"
    variants = en_to_ru.get(str(text).strip().lower(), [])
    # уникальные непустые
    variants = sorted({v.strip() for v in variants if v and v.strip()})
    if len(variants) == 0:
        return "(перевод не найден)"
    if len(variants) > 1:
        return "(требуется ручная проверка)"
    return variants[0]

print(f"Dictionary entries: {len(raw_dict)}")
print(f"Unique EN keys: {len(en_to_ru)}")

Dictionary entries: 116330
Unique EN keys: 95870


Ячейка 4 — Загрузка structured data

In [3]:
# %%
"""
LOAD STRUCTURED
+ поправка лейбла part_mag_05_borg из УЖЕ существующей Ripper-строки map
"""
with open(STRUCTURED / "compositions" / "all.json", encoding="utf-8") as f:
    compositions = json.load(f)

bosses_path = STRUCTURED / "bosses" / "all.json"
handle_to_boss: dict[str, dict] = {}
bosses_list = []
if bosses_path.exists():
    with open(bosses_path, encoding="utf-8") as f:
        bosses_list = json.load(f)
    for b in bosses_list:
        for h in b.get("dedicated_handles") or []:
            hl = str(h).lower()
            handle_to_boss[hl] = {
                "boss_key": b.get("boss_key"),
                "display_name": b.get("display_name"),
                "display_guid": b.get("display_guid"),
            }
            if "." in hl:
                handle_to_boss[hl.split(".", 1)[-1]] = handle_to_boss[hl]

part_name_eng_map: dict[str, str] = {}
pmap_path = STRUCTURED / "parts" / "part_name_eng_map.json"
if pmap_path.exists():
    with open(pmap_path, encoding="utf-8") as f:
        part_name_eng_map = json.load(f)
else:
    plab_path = STRUCTURED / "parts" / "part_uistat_labels.json"
    if plab_path.exists():
        with open(plab_path, encoding="utf-8") as f:
            for row in json.load(f):
                if row.get("name_eng") and row.get("part"):
                    part_name_eng_map[row["part"]] = row["name_eng"]

def _ripper_mag_label(nmap: dict) -> str | None:
    """Берём строку, которая УЖЕ есть в map. Ничего не выдумываем."""
    for _k, e in (nmap or {}).items():
        el = (e or "").lower()
        if "ripper" in el and "charges before" in el and "full auto" in el:
            return e
    return None

_rip = _ripper_mag_label(part_name_eng_map)
_n_remap = 0
if _rip:
    for k, e in list(part_name_eng_map.items()):
        if "part_mag_05_borg" not in k.lower():
            continue
        el = (e or "").lower()
        if "order" in el or "multiple rounds" in el:
            part_name_eng_map[k] = _rip
            _n_remap += 1
            print(f"remap {k} → {_rip[:70]}")

print(f"Compositions loaded: {len(compositions)}")
print("Rarity counts:", Counter(c.get("rarity") for c in compositions))
print(f"Bosses: {len(bosses_list)} | handle map: {len(handle_to_boss)}")
print(f"Part name_eng map: {len(part_name_eng_map)} | borg remaps: {_n_remap}")

remap part_mag_05_borg → Ripper - This Gun charges before Full Auto firing
Compositions loaded: 245
Rarity counts: Counter({'legendary': 238, 'pearlescent': 6, 'uncommon': 1})
Bosses: 82 | handle map: 396
Part name_eng map: 128 | borg remaps: 1


Ячейка 5 — Фильтр пушек legendary + pearlescent

In [4]:
# %%
"""
FILTER: guns PS/SR/AR/SG/SM + legendary/pearlescent
"""
def detect_type_and_manu(c: dict) -> tuple[str | None, str | None]:
    """Возвращает (TYPE, MANU) или (None, None). Без догадок."""
    # 1) из item_types вида JAK_PS
    for it in c.get("item_types") or []:
        parts = str(it).upper().split("_")
        if len(parts) >= 2 and parts[0] in MANU_RU and parts[1] in GUN_TYPES:
            return parts[1], parts[0]
        if len(parts) >= 2 and parts[1] in MANU_RU and parts[0] in GUN_TYPES:
            return parts[0], parts[1]

    # 2) из composition: ord_sr.comp_05_...
    comp = (c.get("composition") or "").lower()
    m = re.match(r"^([a-z]+)_([a-z]{2})\.", comp)
    if m:
        manu_c, typ_c = m.group(1).upper(), m.group(2).upper()
        if manu_c in MANU_RU and typ_c in GUN_TYPES:
            return typ_c, manu_c

    # 3) basetags / uni — только если явно видно
    return None, None

guns = []
skipped = Counter()

for c in compositions:
    rarity = (c.get("rarity") or "").lower()
    if rarity not in RARITIES:
        skipped["wrong_rarity"] += 1
        continue

    gtype, manu = detect_type_and_manu(c)
    if not gtype or gtype not in GUN_TYPES:
        skipped["not_gun_type"] += 1
        continue
    if not manu or manu not in MANU_RU:
        skipped["no_manu"] += 1
        continue

    guns.append({
        "raw": c,
        "type_code": gtype,
        "manu_code": manu,
    })

print(f"Guns selected: {len(guns)}")
print("Skipped:", dict(skipped))
print("By type:", Counter(g["type_code"] for g in guns))
print("By manu:", Counter(g["manu_code"] for g in guns))

Guns selected: 145
Skipped: {'not_gun_type': 99, 'wrong_rarity': 1}
By type: Counter({'AR': 34, 'SG': 33, 'PS': 28, 'SR': 26, 'SM': 24})
By manu: Counter({'JAK': 24, 'DAD': 22, 'MAL': 18, 'VLA': 18, 'BOR': 16, 'TOR': 16, 'ORD': 16, 'TED': 15})


Ячейка 6 — Сбор всех слотов (для колонок parts)

In [5]:
# %%
"""
COLLECT ALL PART SLOTS
не берём выключенные (max==0) и пустые inherit
"""
RESERVED_SLOT_NAMES = {
    "of_game", "of_game_eng", "of_game_ru",
    "part_of_game", "part_of_game_eng", "part_of_game_ru",
}

all_slots = set()
n_skip_off = n_skip_empty = 0
for g in guns:
    slots = g["raw"].get("slots") or {}
    for slot, info in slots.items():
        if not slot or str(slot).lower() in RESERVED_SLOT_NAMES:
            continue
        if f"part_{slot}" in {
            "part_of_game", "part_of_game_eng", "part_of_game_ru",
            "part_source", "part_source_eng", "part_source_ru",
        }:
            continue
        if isinstance(info, dict):
            if info.get("max") == 0:
                n_skip_off += 1
                continue
            if info.get("inherit_from_type") and not (info.get("parts") or []):
                n_skip_empty += 1
                continue
        all_slots.add(slot)

all_slots = sorted(all_slots)
print(f"Unique part slots: {len(all_slots)}")
print(all_slots)
print(f"skipped max==0: {n_skip_off} | skipped empty inherit: {n_skip_empty}")

Unique part slots: 19
['barrel', 'barrel_acc', 'body_acc', 'body_ele', 'body_mag', 'foregrip', 'grip', 'hyperion_secondary_acc', 'magazine', 'magazine_acc', 'magazine_ted_thrown', 'scope', 'scope_acc', 'secondary_ammo', 'secondary_ele', 'tediore_acc', 'tediore_secondary_acc', 'underbarrel', 'underbarrel_acc']
skipped max==0: 38 | skipped empty inherit: 307


Ячейка 7 — Сборка строк DataFrame

In [6]:
# %%
"""
BUILD ROWS
колонки производителей: Производитель(технический_ключ)
лицензия — только по ключу детали; иначе производитель пушки
"""
rows = []

LICENSED_KEY_RE = re.compile(
    r"licensed|part_shield_|mag_04_cov|mag_05_borg|malswitch|"
    r"secondary_elem|secondary_ammo_|part_multi_ted",
    re.I,
)
LICENSED_ENG_RE = re.compile(
    r"licensed|torgue -|hyperion-|jakobs-|tediore-|atlas-|"
    r"daedalus-|ripper-|cov-|maliwan -",
    re.I,
)

def format_one_part(p: str) -> str:
    if not p:
        return ""
    eng = part_name_eng_map.get(p) or part_name_eng_map.get(str(p))
    if not eng:
        return p
    pl = str(p).lower()
    if LICENSED_KEY_RE.search(pl):
        return eng
    if LICENSED_ENG_RE.search(eng):
        return p
    return eng

def format_part_list(parts: list) -> str:
    if not parts:
        return "-"
    out = [format_one_part(p) for p in parts if p]
    return ", ".join(out) if out else "-"

MANU_PART_COLS = [
    "Ripper",
    "COV",
    "Torgue",
    "Torgue (гир)",
    "Torgue (лип)",
    "Jakobs",
    "Tediore",
    "Hyperion (щит)",
    "Hyperion (стаб)",
    "Atlas",
    "Daedalus",
    "Maliwan",
    "Vladof",
    "Order",
]

GUN_MANU_TO_COL = {
    "BOR": "Ripper",
    "DAD": "Daedalus",
    "JAK": "Jakobs",
    "MAL": "Maliwan",
    "TED": "Tediore",
    "TOR": "Torgue",
    "VLA": "Vladof",
    "ORD": "Order",
}

def buckets_for_part(part: str, gun_manu: str) -> list[str]:
    """Только ключ детали. Без UI-текста."""
    pl = (part or "").lower()
    found = []

    if "part_mag_05_borg" in pl or "lp_bor_mag" in pl or "licensed_bor" in pl:
        found.append("Ripper")
    if "part_mag_04_cov" in pl or "lp_cov_mag" in pl or "licensed_cov" in pl:
        found.append("COV")

    is_sticky = bool(re.search(r"tor_sticky|torgue_sticky|sticky", pl)) and bool(
        re.search(r"torgue|tor_|mag_torgue", pl)
    )
    is_gyro = bool(re.search(r"tor_gyro|torgue_normal|mag_torgue_normal", pl))
    if is_sticky:
        found.append("Torgue (лип)")
    elif is_gyro:
        found.append("Torgue (гир)")

    if "licensed_jak" in pl or "lp_jak" in pl or "jak_ricochet" in pl:
        found.append("Jakobs")
    if (
        "licensed_ted" in pl
        or "part_multi_ted" in pl
        or re.search(r"ted_combo|ted_mirv|ted_shooting|ted_replicator|ted_multimod", pl)
    ):
        found.append("Tediore")
    if "part_shield_" in pl or "hyp_shield" in pl or "licensed_hyp" in pl:
        found.append("Hyperion (щит)")
    if "lp_hyp_acc" in pl or re.search(r"grip_.*_hyp|grip_04_hyp|hyp_acc", pl):
        found.append("Hyperion (стаб)")
    if "lp_atlas" in pl or "underbarrel_04_atlas" in pl or "licensed_atlas" in pl:
        found.append("Atlas")
    if "lp_dad_" in pl or "licensed_dad" in pl or "secondary_ammo" in pl:
        found.append("Daedalus")
    if "malswitch" in pl or "secondary_elem" in pl or "licensed_mal" in pl or "lp_mal_" in pl:
        found.append("Maliwan")

    # licensed_multi без ted/atlas в ключе — не угадываем производителя
    if "part_barrel_licensed_multi" in pl and not found:
        pass

    if found:
        return found

    native = GUN_MANU_TO_COL.get((gun_manu or "").upper())
    return [native] if native else []

def sources_from_comp(c: dict) -> tuple[str, str, str]:
    drop_sources = c.get("drop_sources") or []
    if not drop_sources:
        return "-", "-", "-"
    tech_eng, name_engs, name_rus = [], [], []
    seen_boss = set()
    for s in drop_sources:
        key = (s.get("key") if isinstance(s, dict) else str(s)) or ""
        nice = key
        for prefix in ("itempoollist_", "ItemPoolList_"):
            if nice.lower().startswith(prefix.lower()):
                nice = nice[len(prefix):]
        if isinstance(s, dict) and s.get("is_trueboss") and not nice.lower().endswith("_trueboss"):
            nice = f"{nice} (TrueBoss)"
        tech_eng.append(nice)
        handle = (c.get("composition") or "").lower()
        boss = handle_to_boss.get(handle)
        if not boss:
            bk = re.sub(r"_trueboss$|_true$", "", nice, flags=re.I).lower()
            for b in bosses_list:
                if (b.get("boss_key") or "").lower() == bk:
                    boss = {
                        "display_name": b.get("display_name"),
                        "display_guid": b.get("display_guid"),
                    }
                    break
        if boss and boss.get("display_name"):
            dn = boss["display_name"]
            if dn not in seen_boss:
                seen_boss.add(dn)
                name_engs.append(dn)
                guid = boss.get("display_guid")
                ru = translate_by_guid(guid) if guid else translate_by_en(dn)
                if ru in ("(перевод не найден)", "-") and dn:
                    ru = translate_by_en(dn)
                name_rus.append(ru)
    return (
        ", ".join(tech_eng) if tech_eng else "-",
        ", ".join(name_engs) if name_engs else "-",
        ", ".join(name_rus) if name_rus else "-",
    )

for g in guns:
    c = g["raw"]
    gtype = g["type_code"]
    manu = g["manu_code"]

    item_id = c.get("composition") or c.get("internal_name") or "-"
    name_eng = c.get("display_name") or "-"
    name_ru = translate_by_guid(c.get("display_guid")) if c.get("display_guid") else (
        translate_by_en(name_eng) if name_eng != "-" else "-"
    )
    if name_ru == "(перевод не найден)" and name_eng != "-":
        name_ru = translate_by_en(name_eng)

    rarity_ru = RARITY_RU.get(c.get("rarity"), "-")
    type_ru = TYPE_RU.get(gtype, "-")
    manu_ru = MANU_RU.get(manu, "-")

    rt = c.get("red_text") or {}
    red_eng = (rt.get("text") if isinstance(rt, dict) else None) or "-"
    red_guid = rt.get("guid") if isinstance(rt, dict) else None
    if red_guid:
        red_ru = translate_by_guid(red_guid)
        if red_ru == "(перевод не найден)" and red_eng != "-":
            red_ru = translate_by_en(red_eng)
    else:
        red_ru = translate_by_en(red_eng) if red_eng != "-" else "-"

    fx = c.get("legendary_effect") or {}
    fx_eng = (fx.get("text") if isinstance(fx, dict) else None) or "-"
    fx_guid = fx.get("guid") if isinstance(fx, dict) else None
    if fx_guid:
        fx_ru = translate_by_guid(fx_guid)
        if fx_ru == "(перевод не найден)" and fx_eng != "-":
            fx_ru = translate_by_en(fx_eng)
    else:
        fx_ru = translate_by_en(fx_eng) if fx_eng != "-" else "-"

    phosphene = "есть" if c.get("has_phosphene") else "нет"
    world_drop = "есть" if c.get("has_world_drop") else "нет"

    signals = c.get("origin_signals") or []
    drop_sources = c.get("drop_sources") or []
    if signals:
        game_part_eng = ", ".join(signals)
        game_part_ru = ", ".join(
            s if translate_by_en(s) in ("(перевод не найден)", "(требуется ручная проверка)")
            else translate_by_en(s)
            for s in signals
        )
    elif drop_sources:
        game_part_eng = "Base Game"
        game_part_ru = "Основная игра"
    else:
        game_part_eng = "-"
        game_part_ru = "-"

    source_eng, source_name_eng, source_name_ru = sources_from_comp(c)

    elems = c.get("elements") or []
    elements_ru = "-" if not elems else ", ".join(ELEMENT_RU.get(e, "(unknown)") for e in elems)

    row = {
        "item_id": item_id,
        "name_eng": name_eng,
        "name_ru": name_ru,
        "rarity": rarity_ru,
        "type": type_ru,
        "manufacturer": manu_ru,
        "red_text_eng": red_eng,
        "red_text_ru": red_ru,
        "legendary_effect_eng": fx_eng,
        "legendary_effect_ru": fx_ru,
        "phosphene": phosphene,
        "world_drop": world_drop,
        "game_part_eng": game_part_eng,
        "game_part_ru": game_part_ru,
        "source_eng": source_eng,
        "source_name_eng": source_name_eng,
        "source_name_ru": source_name_ru,
        "elements": elements_ru,
    }

    slots = c.get("slots") or {}
    for slot in all_slots:
        col_name = f"part_{slot}"
        if col_name in ("part_of_game", "part_of_game_eng", "part_of_game_ru"):
            continue
        info = slots.get(slot) or {}
        if isinstance(info, dict) and info.get("max") == 0:
            row[col_name] = "-"
            continue
        row[col_name] = format_part_list(info.get("parts") or [])

    by_manu = {col: [] for col in MANU_PART_COLS}
    seen = set()
    for slot, info in slots.items():
        if isinstance(info, dict) and info.get("max") == 0:
            continue
        parts = (info.get("parts") if isinstance(info, dict) else []) or []
        for p in parts:
            if not p:
                continue
            label = f"{p}"
            for bucket in buckets_for_part(p, manu):
                cell = f"{bucket}({p})"
                key = (bucket, cell)
                if key in seen:
                    continue
                seen.add(key)
                by_manu[bucket].append(cell)

    for col in MANU_PART_COLS:
        row[col] = ", ".join(by_manu[col]) if by_manu[col] else "-"

    rows.append(row)

print(f"Rows built: {len(rows)}")
filled = {c: 0 for c in MANU_PART_COLS}
for r in rows:
    for col in MANU_PART_COLS:
        if r.get(col) and r[col] != "-":
            filled[col] += 1
print("Guns with ≥1 part in manufacturer col:")
for col in MANU_PART_COLS:
    print(f"  {col}: {filled[col]}")

Rows built: 145
Guns with ≥1 part in manufacturer col:
  Ripper: 64
  COV: 50
  Torgue: 16
  Torgue (гир): 14
  Torgue (лип): 12
  Jakobs: 48
  Tediore: 30
  Hyperion (щит): 26
  Hyperion (стаб): 9
  Atlas: 23
  Daedalus: 24
  Maliwan: 39
  Vladof: 18
  Order: 16


Ячейка 8 — DataFrame + экспорт CSV

In [7]:
# %%
"""
DATAFRAME + EXPORT
"""
df = pd.DataFrame(rows)

MANU_PART_COLS = [
    "Ripper",
    "COV",
    "Torgue",
    "Torgue (гир)",
    "Torgue (лип)",
    "Jakobs",
    "Tediore",
    "Hyperion (щит)",
    "Hyperion (стаб)",
    "Atlas",
    "Daedalus",
    "Maliwan",
    "Vladof",
    "Order",
]

base_cols = [
    "item_id", "name_eng", "name_ru", "rarity", "type", "manufacturer",
    "red_text_eng", "red_text_ru",
    "legendary_effect_eng", "legendary_effect_ru",
    "phosphene", "world_drop",
    "game_part_eng", "game_part_ru",
    "source_eng",
    "source_name_eng", "source_name_ru",
    "elements",
]
part_cols = [
    c for c in df.columns
    if c.startswith("part_") and c not in (
        "part_of_game", "part_of_game_eng", "part_of_game_ru",
    )
]
manu_cols = [c for c in MANU_PART_COLS if c in df.columns]
df = df[base_cols + sorted(part_cols) + manu_cols]

now = datetime.now()
fname = f"bl4_guns_{now.strftime('%m-%d-%Y_%H-%M-%S')}.csv"
out_path = OUT_DIR / fname
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"Saved: {out_path.resolve()}")
print(f"Shape: {df.shape}")
print()
qa = ["item_id", "name_eng", "Daedalus", "Ripper", "COV", "Torgue", "Torgue (гир)", "Torgue (лип)", "Jakobs"]
print(df[[c for c in qa if c in df.columns]].head(8).to_string())

Saved: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/output/bl4_guns_08-22-2026_19-38-59.csv
Shape: (145, 51)

                          item_id          name_eng                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          Daedalus                                                                                                                                                                                                                         Ripper                   COV                                                               

Ячейка 9 — Быстрая проверка качества

In [8]:
# %%
"""
QA CHECK
"""
print("=== QA ===")
print("name_ru value counts (top):")
print(df["name_ru"].value_counts().head(10))
print()
print("red_text_ru '(перевод не найден)':", (df["red_text_ru"] == "(перевод не найден)").sum())
print("name_ru '(перевод не найден)':", (df["name_ru"] == "(перевод не найден)").sum())
print()
print("rarity:", df["rarity"].value_counts().to_dict())
print("type:", df["type"].value_counts().to_dict())
print("manufacturer:", df["manufacturer"].value_counts().to_dict())

=== QA ===
name_ru value counts (top):
name_ru
ПУЛЕМЕТ АТЛИНГА     8
-                   5
ПАРАЗИТ             2
ОБИТАТЕЛЬ БЕЗДНЫ    1
АРК-ТАНГ            1
БОГ БЮДЖЕТА         1
ПОДЪЕМ АШЕРОВ       1
КОРПУС              1
ОПАСНЫЙ ОКСИБЕЛ     1
БУМСЛАНГ            1
Name: count, dtype: int64

red_text_ru '(перевод не найден)': 0
name_ru '(перевод не найден)': 0

rarity: {'Легендарный': 139, 'Перламутровый': 6}
type: {'Штурмовая винтовка': 34, 'Дробовик': 33, 'Пистолет': 28, 'Снайперская винтовка': 26, 'Пистолет-пулемёт': 24}
manufacturer: {'Джейкобс': 24, 'Дедалус': 22, 'Маливань': 18, 'Владоф': 18, 'Риппер': 16, 'Торг': 16, 'Орден': 16, 'Тедиор': 15}
